
# Update Categories — LLM-assisted vault enrichment

Spójrz na wyciągnięte intencje dla Q04 w logach:
surfaces: [blaty, umywalki, marmur, łazienki]
categories: [odkamienianie, usuwanie-kamienia, plyny-do-lazienki, srodki-kwasowe]

Diagnoza: Produkt 11 (Zurada Czysta Łazienka) najwyraźniej nie jest podlinkowany w Twoim Vaulcie (plikach Markdown) pod żadną z tych kategorii/powierzchni! To jest największa wada architektury opartej na grafie wiedzy: jeśli w pliku .md kategorii plyny-do-lazienki lub powierzchni umywalki nie ma linku [[Produkty/Zurada Czysta Łazienka]], to funkcja lookup_candidates w ogóle nie przekaże tego produktu do LLMa w Kroku 4. LLM go nie poleci, bo go nie widzi.

For each product in `zurada_vault/Produkty/`, use an LLM to select the best matching categories from all 285 available in `zurada_vault/Kategorie/`, then write the changes back to the vault.

**What gets updated:**
- `kategorie: [...]` in product frontmatter
- `**Kategorie:** [[...]]` links in product body
- `liczba_produktow:` in each category frontmatter
- `## Produkty (N)` list in each category body

**Workflow (run cells top to bottom):**
1. Load vault
2. Define pydantic model with `CategoryEnum` — closed set, instructor retries on invalid names
3. Demo on a few products
4. Process all products → save diffs to `new_categories.json`
5. Apply to product files
6. Rebuild category files




In [6]:
%pip install instructor -q

Note: you may need to restart the kernel to use updated packages.


In [7]:
import os
import re
import json
import time
import enum as _enum
from pathlib import Path
from typing import List
from datetime import datetime

from openai import OpenAI
from pydantic import BaseModel, Field
import instructor

MODEL    = "openai/gpt-5.5"
BASE_URL = "https://openrouter.ai/api/v1"
OPENROUTER_API_KEY = os.environ.get("OPENROUTER_API_KEY", "Brak klucza")

VAULT_DIR    = Path("zurada_vault")
DRY_RUN_FILE = Path("new_categories.json")

_raw_client = OpenAI(api_key=OPENROUTER_API_KEY, base_url=BASE_URL)
client = instructor.patch(_raw_client, mode=instructor.Mode.MD_JSON)

print(f"Model:   {MODEL}")
print(f"Vault:   {VAULT_DIR}")
print(f"API key: {'OK' if OPENROUTER_API_KEY != 'Brak klucza' else 'BRAK'}")

Model:   openai/gpt-5.5
Vault:   zurada_vault
API key: OK


---
## Step 1 — Load vault

In [8]:
def _opis(path: Path) -> str:
    """Extract ## Opis paragraph from a vault .md page."""
    text = path.read_text(encoding="utf-8")
    m = re.search(r'## Opis\s*\n+(.+?)(?:\n\n##|\Z)', text, re.DOTALL)
    return m.group(1).strip()[:200] if m else ""

def _frontmatter_field(text: str, field: str) -> str:
    m = re.search(rf'^{field}:\s*(.+)$', text, re.MULTILINE)
    return m.group(1).strip() if m else ""

def _current_categories(text: str) -> list[str]:
    m = re.search(r'^kategorie:\s*(\[.*?\])', text, re.MULTILINE)
    if not m:
        return []
    try:
        return json.loads(m.group(1))
    except Exception:
        return []


# categories: name → description
categories: dict[str, str] = {}
for f in sorted((VAULT_DIR / "Kategorie").glob("*.md")):
    categories[f.stem] = _opis(f)

# products: stem → path
products: dict[str, Path] = {}
for f in sorted((VAULT_DIR / "Produkty").glob("*.md")):
    products[f.stem] = f

print(f"Categories loaded: {len(categories)}")
print(f"Products loaded:   {len(products)}")
print()
print("Sample categories:")
for name, desc in list(categories.items())[:5]:
    print(f"  {name}: {desc[:80]}")

Categories loaded: 285
Products loaded:   262

Sample categories:
  aerozole-techniczne: Aerozole techniczne są przeznaczone dla serwisów, warsztatów i działów utrzymani
  akcesoria-do-czyszczenia: Akcesoria do czyszczenia obejmują narzędzia pomocnicze dla użytkowników domowych
  akcesoria-do-mycia: Akcesoria do mycia są stosowane przy ręcznym lub wspomaganym myciu pojazdów, pow
  akcesoria-do-sprzatania: Akcesoria do sprzątania są przeznaczone dla ekip porządkowych, biur, zakładów us
  aktywna-piana: Aktywna piana to środek do mycia wstępnego pojazdów i dużych powierzchni, najczę


---
## Step 2 — Build CategoryEnum and pydantic model

`CategoryEnum` gives the LLM a closed set of valid names. Pydantic rejects anything outside it; instructor retries automatically with the validation error.

In [9]:
CategoryEnum = _enum.Enum(
    "CategoryEnum",
    {k.upper().replace(" ", "_").replace("-", "_").replace("(", "").replace(")", ""): k
     for k in sorted(categories.keys())},
    type=str,
)

# Top-30 categories by liczba_produktow → hints in Field description
def _count(path: Path) -> int:
    m = re.search(r'liczba_produktow:\s*(\d+)', path.read_text(encoding="utf-8"))
    return int(m.group(1)) if m else 0

_top30 = sorted((VAULT_DIR / "Kategorie").glob("*.md"), key=_count, reverse=True)[:30]
_cat_hint = "; ".join(f"{f.stem}: {categories[f.stem][:60]}" for f in _top30)


class CategorySelection(BaseModel):
    categories: List[CategoryEnum] = Field(
        ...,
        description=(
            "3 do 8 kategorii najlepiej pasujących do produktu. "
            "Uwzględnij: typ produktu (żel/pianka/pasta), przeznaczenie (domowe/profesjonalne), "
            "branżę docelową, akcję chemiczną (odkamienianie, dezynfekcja, odtłuszczanie). "
            "Domyślnie wybierz najbardziej trafne — nie dodawaj kategorii 'na wszelki wypadek'. "
            f"Najczęstsze kategorie w vaulcie: {_cat_hint}"
        ),
    )
    reasoning: str = Field(default="", description="Krótkie uzasadnienie")


print(f"CategoryEnum:      {len(list(CategoryEnum))} valid values")
print("CategorySelection: ready")

CategoryEnum:      285 valid values
CategorySelection: ready


---
## Step 3 — LLM function + demo

In [10]:
SYSTEM_PROMPT = """Jesteś ekspertem ds. chemii gospodarczej i profesjonalnej firmy Zurada.
Twoim zadaniem jest przypisać trafne kategorie do produktu czyszczącego na podstawie jego opisu.

Zasady:
1. Wybierz 3-8 kategorii, które najlepiej opisują ten produkt.
2. Uwzględnij typ produktu, przeznaczenie (dom vs przemysł/horeca), branżę i właściwości aktywne.
3. Preferuj kategorie szczegółowe (np. 'zele-do-wc') nad ogólnymi (np. 'czyszczenie').
4. Nie wybieraj kategorii tylko dlatego, że brzmią podobnie — muszą faktycznie pasować."""


def select_categories(product_name: str, product_text: str) -> CategorySelection:
    return client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user",   "content": f"Produkt: {product_name}\n\n{product_text}"},
        ],
        response_model=CategorySelection,
        max_tokens=5000,
        temperature=0,
    )


# Demo on 3 products
for demo_name in list(products.keys())[:3]:
    text    = products[demo_name].read_text(encoding="utf-8")
    current = _current_categories(text)
    result  = select_categories(demo_name, text)
    new     = [c.value for c in result.categories]

    added   = sorted(set(new) - set(current))
    removed = sorted(set(current) - set(new))

    print(f"\n{demo_name}")
    print(f"  Current:  {current}")
    print(f"  New:      {new}")
    if added:   print(f"  + added:  {added}")
    if removed: print(f"  - removed:{removed}")
    print(f"  Reason:   {result.reasoning[:100]}")
    time.sleep(0.5)


Zurada Agro Kompleks
  Current:  ['rolnictwo', 'mycie-maszyn', 'chemia-do-maszyn-rolniczych', 'zestawy-czyszczace']
  New:      ['zestawy-czyszczace', 'chemia-do-maszyn-rolniczych', 'mycie-maszyn', 'maszyny-rolnicze', 'rolnictwo', 'aktywna-piana', 'odtluszczacze', 'hydrowoski']
  + added:  ['aktywna-piana', 'hydrowoski', 'maszyny-rolnicze', 'odtluszczacze']
  Reason:   Produkt jest zestawem do kompleksowego mycia, odtłuszczania i zabezpieczania maszyn rolniczych. Zawi

Zurada Agro Piana
  Current:  ['rolnictwo', 'mycie-maszyn', 'piany-aktywne', 'chemia-profesjonalna']
  New:      ['chemia-do-maszyn-rolniczych', 'maszyny-rolnicze', 'rolnictwo', 'mycie-maszyn', 'piany-aktywne', 'mycie-cisnieniowe', 'chemia-profesjonalna']
  + added:  ['chemia-do-maszyn-rolniczych', 'maszyny-rolnicze', 'mycie-cisnieniowe']
  Reason:   Koncentrat aktywnej piany przeznaczony do pianowego, natryskowego i ciśnieniowego mycia maszyn rolni

Zurada Aktywna Piana
  Current:  ['nan']
  New:      ['aktywna-piana',

---
## Step 4 — Process all products (calls LLM, shows diffs, saves to `new_categories.json`)

This is the expensive step (~262 LLM calls, ~5–15 min). Run once, review the output, then use the apply cells below.

In [11]:
new_categories: dict[str, list[str]] = {}
changed_count = 0
error_count   = 0

for i, (name, path) in enumerate(products.items(), 1):
    text    = path.read_text(encoding="utf-8")
    current = _current_categories(text)

    try:
        result = select_categories(name, text)
        new    = [c.value for c in result.categories]
        new_categories[name] = new

        added   = sorted(set(new) - set(current))
        removed = sorted(set(current) - set(new))
        changed = bool(added or removed)

        if changed:
            changed_count += 1
            print(f"[CHANGE] {name}")
            if added:   print(f"         + {added}")
            if removed: print(f"         - {removed}")

    except Exception as e:
        error_count += 1
        print(f"[ERROR ] {name}: {e}")
        new_categories[name] = current  # keep existing on error

    if i % 20 == 0:
        print(f"--- {i}/{len(products)} done ---")

    time.sleep(0.3)

# Save for apply cells
with open(DRY_RUN_FILE, "w", encoding="utf-8") as f:
    json.dump(
        {"timestamp": datetime.now().isoformat(), "products": new_categories},
        f, ensure_ascii=False, indent=2,
    )

print(f"\n{'='*60}")
print(f"Products processed: {len(new_categories)}")
print(f"Changed:            {changed_count}")
print(f"Errors (kept old):  {error_count}")
print(f"Saved to:           {DRY_RUN_FILE}")

[CHANGE] Zurada Agro Kompleks
         + ['aktywna-piana', 'hydrowoski', 'maszyny-rolnicze', 'odtluszczacze']
[CHANGE] Zurada Agro Piana
         + ['chemia-do-maszyn-rolniczych', 'chemia-rolnicza', 'maszyny-rolnicze']
[CHANGE] Zurada Aktywna Piana
         + ['aktywna-piana', 'chemia-samochodowa', 'maszyny-budowlane', 'maszyny-rolnicze', 'mycie-bezdotykowe', 'mycie-cisnieniowe', 'mycie-pojazdow-ciezarowych', 'mycie-zewnetrzne-samochodu']
         - ['nan']
[CHANGE] Zurada Alka Czystość
         + ['chemia-przemyslowa', 'czyszczenie-maszyn', 'higiena-urzadzen', 'mycie-cisnieniowe', 'mycie-przemyslowe', 'preparaty-alkaliczne']
[CHANGE] Zurada Alka Degreaser
         + ['mycie-i-odtluszczanie', 'preparaty-alkaliczne']
[CHANGE] Zurada Alka Floor
         + ['odtluszczacze', 'preparaty-alkaliczne']
[CHANGE] Zurada Alka Moc
         + ['chemia-profesjonalna', 'odtluszczacze', 'posadzki-przemyslowe', 'preparaty-alkaliczne']
         - ['chemia-samochodowa', 'firmy-sprzatajace', 'horeca-hotel

---
## Step 5 — Apply: update product files

Reads `new_categories.json`, rewrites `kategorie:` frontmatter and `**Kategorie:**` links in each product `.md`.

In [15]:
with open(DRY_RUN_FILE, encoding="utf-8") as f:
    saved = json.load(f)
new_categories = saved["products"]
print(f"Loaded {len(new_categories)} products from {DRY_RUN_FILE} (saved {saved['timestamp']})")


def update_product_file(path: Path, new_cats: list[str]) -> None:
    text = path.read_text(encoding="utf-8")

    # 1. frontmatter: kategorie: [...]
    cats_str = json.dumps(new_cats, ensure_ascii=False)
    text = re.sub(
        r'^kategorie:\s*\[.*?\]',
        f'kategorie: {cats_str}',
        text,
        flags=re.MULTILINE,
    )

    # 2. body: **Kategorie:** [[Kategorie/x|x]] · ...
    links = " \u00b7 ".join(f"[[Kategorie/{c}|{c}]]" for c in new_cats)
    text = re.sub(
        r'^\*\*Kategorie:\*\*.*$',
        f'**Kategorie:** {links}',
        text,
        flags=re.MULTILINE,
    )

    path.write_text(text, encoding="utf-8")


written = 0
for name, new_cats in new_categories.items():
    if name in products:
        update_product_file(products[name], new_cats)
        written += 1

print(f"Updated {written} product files")

Loaded 262 products from new_categories.json (saved 2026-05-27T08:36:06.709061)
Updated 262 product files


---
## Step 6 — Rebuild category files

Builds reverse index (category → products), then rewrites `liczba_produktow:` and `## Produkty (N)` in each category `.md`.

In [16]:
# Build reverse index
cat_to_products: dict[str, list[str]] = {cat: [] for cat in categories}
for name, new_cats in new_categories.items():
    for cat in new_cats:
        if cat in cat_to_products:
            cat_to_products[cat].append(name)


def update_category_file(cat_name: str, product_names: list[str]) -> None:
    path = VAULT_DIR / "Kategorie" / f"{cat_name}.md"
    if not path.exists():
        return
    text = path.read_text(encoding="utf-8")
    n = len(product_names)

    # frontmatter: liczba_produktow:
    text = re.sub(
        r'^liczba_produktow:\s*\d+',
        f'liczba_produktow: {n}',
        text,
        flags=re.MULTILINE,
    )

    # ## Produkty section (always last section — replace to end of file)
    products_block = (
        f"## Produkty ({n})\n\n"
        + "\n".join(f"- [[Produkty/{p}|{p}]]" for p in sorted(product_names))
    )
    text = re.sub(
        r'## Produkty \(\d+\).*',
        products_block,
        text,
        flags=re.DOTALL,
    )

    path.write_text(text, encoding="utf-8")


updated = 0
empty   = 0
for cat_name, product_names in cat_to_products.items():
    update_category_file(cat_name, product_names)
    updated += 1
    if not product_names:
        empty += 1

print(f"Rebuilt {updated} category files")
print(f"Empty categories (0 products): {empty}")

Rebuilt 285 category files
Empty categories (0 products): 1


---
## Summary

In [17]:
from collections import Counter

cat_counts  = Counter(cat for cats in new_categories.values() for cat in cats)
avg_per_product = sum(len(v) for v in new_categories.values()) / len(new_categories)
cats_used   = sum(1 for c in cat_counts if cat_counts[c] > 0)

print(f"Products updated:         {len(new_categories)}")
print(f"Avg categories/product:   {avg_per_product:.1f}")
print(f"Categories with products: {cats_used}/{len(categories)}")
print()
print("Top 15 most assigned categories:")
for cat, count in cat_counts.most_common(15):
    print(f"  {count:3d}  {cat}")

Products updated:         262
Avg categories/product:   6.8
Categories with products: 284/285

Top 15 most assigned categories:
   57  horeca-hotele
   51  kosmetyki-samochodowe
   50  chemia-warsztatowa
   48  chemia-samochodowa
   47  dla-domu
   45  firmy-sprzatajace
   40  odtluszczacze
   36  przemysl-i-warsztat
   36  szkoly
   29  mycie-zewnetrzne-samochodu
   24  higiena-rak
   23  mycie-i-odtluszczanie
   23  mycie-rak
   23  chemia-gospodarcza
   23  mycie-i-pielegnacja-wnetrza


In [19]:
path = VAULT_DIR / "Produkty" / "Zurada Czysta Łazienka.md"
print("teraz widać zmiany w kategoriach dla Zurada Czysta Łazienka.md:")
print(path.read_text(encoding="utf-8"))

teraz widać zmiany w kategoriach dla Zurada Czysta Łazienka.md:
---
product_id: 11
title: "Pianka do sanitariatów Eco"
ph: ""
metody: ["pianowe", "natryskowe", "ręczne"]
certyfikaty: ["ifra", "rspo"]
kategorie: ["higiena-sanitariatow", "czyszczenie-sanitarne", "lazienka", "plyny-do-lazienki", "usuwanie-kamienia", "odkamieniacze", "ekologiczne-produkty-eu-ecolabel", "chemia-gospodarcza"]
---

# Zurada Czysta Łazienka

> Ekologiczny, gotowy do użycia preparat w formie pianki do czyszczenia sanitariatów.

---

**Metody mycia:** [[Metody/pianowe|pianowe]] · [[Metody/natryskowe|natryskowe]] · [[Metody/ręczne|ręczne]]

**Certyfikaty:** [[Certyfikaty/ifra|ifra]] · [[Certyfikaty/rspo|rspo]]

**Dozwolone powierzchnie:** [[Powierzchnie/podłogi|podłogi]] · [[Powierzchnie/wanny|wanny]] · [[Powierzchnie/brodziki|brodziki]] · [[Powierzchnie/kabiny prysznicowe|kabiny prysznicowe]] · [[Powierzchnie/umywalki|umywalki]] · [[Powierzchnie/toalety|toalety]] · [[Powierzchnie/pisuary|pisuary]] · [[Powierzchn